# OCBridge Recruiting Copilot — Interactive Demo

This notebook walks through the pipeline one step at a time so you can inspect each LLM call's output before it feeds the next step. The canonical entrypoint is still `python main.py` — this notebook is a visualization aid, not a replacement.

## Setup

Before running, make sure:
1. You're in the `.venv` virtualenv (VS Code: bottom-right Python selector → `.venv`)
2. Your `ANTHROPIC_API_KEY` is set in the environment

If the key isn't set yet, uncomment the cell below and paste your key (do NOT commit the notebook with the key in it).

In [ ]:
import os


## Imports & setup

In [7]:
import os
import sys
import json
from pathlib import Path

# Hard-coded project root — guaranteed to work in your environment.
PROJECT_ROOT = Path(
    "/home/jovyan/MGTA 495 Marketing/ocbridge/ocbridge_recruiting_copilot"
)
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

from src.llm_client import LLMClient
from src.pipeline import RecruitingPipeline


def show(obj, title=None):
    if title:
        print(f"── {title} " + "─" * (66 - len(title)))
    print(json.dumps(obj, indent=2, ensure_ascii=False))
    print()


llm = LLMClient()
pipeline = RecruitingPipeline(llm)
print(f"✓ Working dir: {PROJECT_ROOT}")
print("✓ LLM client ready, pipeline initialized")


✓ Working dir: /home/jovyan/MGTA 495 Marketing/ocbridge/ocbridge_recruiting_copilot
✓ LLM client ready, pipeline initialized


## Load the Job Description

Defaults to the bundled sample JD. Change `JD_PATH` to use your own.

In [8]:
JD_PATH = Path("sample_data/sample_jd.txt")
jd = JD_PATH.read_text(encoding="utf-8")

# Optional hiring manager notes:
hm_notes = None  # e.g. Path('sample_data/hm_notes.txt').read_text()

print(f"Loaded JD: {JD_PATH}  ({len(jd)} chars)")
print("─" * 70)
print(jd)

Loaded JD: sample_data/sample_jd.txt  (1390 chars)
──────────────────────────────────────────────────────────────────────
Senior Backend Engineer — AI Infrastructure

About us:
We're a Series A startup building the inference layer for production LLM applications. Our customers run mission-critical AI workloads on us and need sub-100ms latency at scale. We're a team of 14, based in San Francisco, profitable, and growing fast.

What you'll do:
- Design and build the core inference routing system that decides which model serves which request
- Own latency-critical paths end to end: from gRPC ingress to GPU scheduling
- Work directly with the founding team on architecture decisions that will define the company for the next 3 years
- Help scale from 10M to 1B requests/day over the next 18 months

You should have:
- 5+ years building backend systems in Python or Go
- Experience with high-throughput, low-latency distributed systems (Kafka, Redis, gRPC, or similar)
- Familiarity with GPU infra

## Step 1 — `extract_jd_signals`

Parses the JD into structured signals: role type, required skills, seniority indicators, company stage, missing info.

In [9]:
signals = pipeline.extract_jd_signals(jd, hm_notes)
show(signals, "JD signals")

── JD signals ────────────────────────────────────────────────────────
{
  "role_type": "Senior Backend Engineer — AI Infrastructure",
  "required_skills": [
    "5+ years building backend systems",
    "Python or Go",
    "High-throughput distributed systems",
    "Low-latency distributed systems",
    "Kafka, Redis, gRPC or similar technologies",
    "gRPC ingress",
    "GPU scheduling"
  ],
  "nice_to_have_skills": [
    "GPU infrastructure",
    "CUDA",
    "Model serving (Triton, vLLM, TGI)",
    "Early-stage startup experience",
    "Open-source contributions to ML infrastructure projects",
    "Observability"
  ],
  "seniority_indicators": [
    "5+ years experience",
    "Own latency-critical paths end to end",
    "Work directly with founding team on architecture decisions",
    "Architecture decisions that will define the company for the next 3 years",
    "Design and build core inference routing system"
  ],
  "company_stage": "Series A",
  "domain": "AI infrastructure",
  "

## Step 2 — `generate_search_strategy`

Turns the structured signals into a sourcing plan: who to target, where to find them, what keywords matter.

Note the seniority edge case: if the model returns an empty/unspecified value, the pipeline substitutes `"Mid to Senior (inferred; JD lacked explicit signal)"` and logs it in the trace. See README for the rationale.

In [10]:
strategy = pipeline.generate_search_strategy(signals)
show(strategy, "Search strategy")

── Search strategy ───────────────────────────────────────────────────
{
  "target_backgrounds": [
    "Senior/Staff backend engineers from GPU cloud providers (CoreWeave, Lambda Labs, RunPod)",
    "Backend infrastructure engineers from ML platform teams at mid-stage startups",
    "Distributed systems engineers with model serving experience at AI companies",
    "IC4-IC5 backend engineers from inference API providers (Replicate, Modal, Baseten)",
    "Infrastructure engineers from ML tooling/MLOps startups (Weights & Biases, Anyscale, Determined AI)",
    "Senior backend engineers who built internal ML serving platforms at tech companies",
    "Early engineers (first 20) from Series A/B infrastructure startups"
  ],
  "target_companies": [
    "OpenAI",
    "Anthropic",
    "Cohere",
    "HuggingFace",
    "Replicate",
    "Modal Labs",
    "Baseten",
    "Anyscale",
    "CoreWeave",
    "Lambda Labs",
    "RunPod",
    "Together AI",
    "Fireworks AI",
    "OctoML",
    "Databricks

## Step 3 — `generate_boolean_query`

Builds a single Boolean query usable on LinkedIn Recruiter or similar tools.

**Bonus — function-style tool call:** after the LLM produces the query, `validate_boolean_query()` runs deterministic checks (balanced parens, AND/OR present, quote balance, length cap) and attaches results to the trace.

In [11]:
boolean_result = pipeline.generate_boolean_query(strategy)
boolean_query = boolean_result["boolean_query"]

print("Boolean query:")
print(boolean_query)
print()
show(boolean_result["_validation"], "Validator output")

Boolean query:
("Senior Backend Engineer" OR "Staff Engineer" OR "Principal Engineer" OR "Backend Engineer") AND ("model serving" OR "inference" OR vLLM OR Triton OR "ML infrastructure" OR "AI infrastructure" OR "GPU infrastructure") AND (Python OR Go OR Golang) AND ("distributed systems" OR Kafka OR gRPC OR "high-throughput" OR microservices) AND (OpenAI OR Anthropic OR Cohere OR HuggingFace OR Replicate OR Modal OR Baseten OR Anyscale OR CoreWeave OR "Lambda Labs" OR RunPod OR "Together AI" OR "Fireworks AI" OR OctoML OR Databricks OR "Scale AI")

── Validator output ──────────────────────────────────────────────────
{
  "is_valid": true,
  "balanced_parens": true,
  "has_and": true,
  "has_or": true,
  "length": 539,
  "warning": ""
}



## Step 4 — `generate_outreach_message` (self-correction loop)

This is the core of the assignment. The agent generates an outreach message and validates it against two criteria:
1. **Length** — strictly under 300 characters
2. **Grounding** — `specific_detail` must be an exact phrase from the JD AND must actually appear in the message

On failure, the agent gets explicit per-failure feedback and retries up to 3 times. After 3 failures it raises `PipelineError` and exits gracefully (with partial trace written to `output.json`).

**Bonus — bias guardrail:** if the passing message contains age-coded, gendered, or other problematic language, a warning is attached. The pipeline flags but does NOT block — a human reviewer makes the final call.

In [12]:
outreach = pipeline.generate_outreach_message(signals, strategy, jd)
show(outreach, "Outreach message")

── Outreach message ──────────────────────────────────────────────────
{
  "outreach_message": "Saw your work on GPU scheduling—we're building the inference routing system for production LLM apps (Series A, 14 people, profitable). Need someone to own latency-critical paths end to end. Open to a quick chat?",
  "specific_detail": "latency-critical paths end to end",
  "character_count": 211,
  "attempts": 1
}



## Step 5 — `generate_candidate_summary`

Produces a mock candidate card. Identity is invented but the reasoning is real — `key_skills` are drawn from the JD signals, `fit_reason` references actual role requirements, and `concerns` surfaces a plausible gap (not a vacuous "no concerns").

In [13]:
summary = pipeline.generate_candidate_summary(
    signals, strategy, boolean_query, outreach
)
show(summary, "Candidate summary")

── Candidate summary ─────────────────────────────────────────────────
{
  "name": "Priya Chandrasekaran",
  "current_company": "Modal Labs",
  "key_skills": [
    "Python backend systems",
    "gRPC microservices",
    "Model serving (vLLM, Triton)",
    "Distributed systems architecture"
  ],
  "fit_reason": "6 years building low-latency inference APIs at Modal and previously Databricks MLflow team. Direct experience with vLLM integration, gRPC ingress design, and high-throughput model routing—exactly the core technical stack for this inference routing system.",
  "concerns": "No visible hands-on GPU scheduling or CUDA-level optimization experience based on public profile. May need to ramp on the lower-level GPU infrastructure pieces that distinguish this role from pure API/serving work."
}



## Pipeline trace

Built incrementally as each step ran. This is the proof the pipeline executes as five distinct steps — Step 4 in particular shows each attempt separately if any retries happened.

In [14]:
show(pipeline.trace, "Pipeline trace")

── Pipeline trace ────────────────────────────────────────────────────
[
  {
    "step": 1,
    "action": "extract_jd_signals",
    "attempt": 1,
    "result": "pass",
    "note": "role_type=Senior Backend Engineer — AI Infrastructure"
  },
  {
    "step": 2,
    "action": "generate_search_strategy",
    "attempt": 1,
    "result": "pass",
    "note": "seniority=Senior to Staff level (5-8+ years). Strong IC4-IC5 contributors who can own critical infrastructure end-to-end and influence architecture. Look for candidates comfortable with early-stage ambiguity and founding-team collaboration typical of Series A environments."
  },
  {
    "step": 3,
    "action": "generate_boolean_query",
    "attempt": 1,
    "result": "pass",
    "note": "chars=539 balanced_parens=True has_and=True has_or=True"
  },
  {
    "step": 4,
    "action": "generate_outreach_message",
    "attempt": 1,
    "result": "pass",
    "note": "chars=211, detail='latency-critical paths end to end'"
  },
  {
    "step": 

## Final output

Assembles the same JSON structure that `python main.py` writes to `output.json`. The top-level keys match the assignment schema exactly: `candidate_search_strategy`, `boolean_query`, `outreach_message`, `candidate_summary`, `pipeline_trace`.

In [15]:
clean_strategy = {k: v for k, v in strategy.items() if not k.startswith("_")}
clean_outreach = {
    "outreach_message": outreach["outreach_message"],
    "specific_detail": outreach["specific_detail"],
    "character_count": outreach["character_count"],
    "attempts": outreach["attempts"],
}

final = {
    "candidate_search_strategy": clean_strategy,
    "boolean_query": boolean_query,
    "outreach_message": clean_outreach,
    "candidate_summary": summary,
    "pipeline_trace": pipeline.trace,
}

# Optional: also write to output.json (uncomment to enable)
# Path('output.json').write_text(json.dumps(final, indent=2, ensure_ascii=False))
# print('Wrote output.json')

show(final, "Final output (matches output.json schema)")

── Final output (matches output.json schema) ─────────────────────────
{
  "candidate_search_strategy": {
    "target_backgrounds": [
      "Senior/Staff backend engineers from GPU cloud providers (CoreWeave, Lambda Labs, RunPod)",
      "Backend infrastructure engineers from ML platform teams at mid-stage startups",
      "Distributed systems engineers with model serving experience at AI companies",
      "IC4-IC5 backend engineers from inference API providers (Replicate, Modal, Baseten)",
      "Infrastructure engineers from ML tooling/MLOps startups (Weights & Biases, Anyscale, Determined AI)",
      "Senior backend engineers who built internal ML serving platforms at tech companies",
      "Early engineers (first 20) from Series A/B infrastructure startups"
    ],
    "target_companies": [
      "OpenAI",
      "Anthropic",
      "Cohere",
      "HuggingFace",
      "Replicate",
      "Modal Labs",
      "Baseten",
      "Anyscale",
      "CoreWeave",
      "Lambda Labs",
      "Ru